# 09 · İçerik Çıkarımının Elle Değerlendirilmesi

Sistemin test setinden 20 belgeden (her türden 5) çıkardığı **110 alan** tek tek gösterilir. Solda belgenin resmi, sağda sistemin çıkardığı alanlar var.
Her alan için bir düğmeye bas, sonra **Kaydet ve sonraki belge**'ye geç.

| Düğme | Ne zaman |
|---|---|
| **Doğru** | Çıkarılan değer belgede yazanla aynı (küçük OCR harf hatası olsa da doğru say) |
| **Yanlış** | Değer hatalı ya da başka bir alanın bilgisi (ör. ad yerine "BIOGRAPHICAL SKETCH") |
| **Kaçırıldı** | Sistem "bulunamadı" dedi ama bilgi belgede var |
| **Belgede yok** | Sistem "bulunamadı" dedi ve bilgi belgede gerçekten yok |

- Seçimler her belgede Drive'a kaydedilir (`bitirme_ocr/cikarim_degerlendirme_v2.csv`). Yarıda bırakırsan tekrar çalıştırınca kaldığın belgeden devam eder.
- En sondaki hücre doğruluk oranlarını hesaplar.
- OCR ya da model gerekmez; sadece Drive'daki resimler kullanılır. Açılış ~30 sn.

Bu 110 alan, raporun **Elle değerlendirme** sekmesindekilerle aynıdır; ikisinden birini doldurman yeterli.

## 1. Drive ve veriler

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, io
import pandas as pd
from PIL import Image

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
OUT = ROOT / "cikarim_degerlendirme_v2.csv"
TR = {"email": "E-posta", "invoice": "Fatura", "resume": "CV", "scientific_publication": "Bilimsel makale"}

In [ ]:
ROWS = [
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "gonderen",
"deger": "Ostern, Eric on Thu, Apr 4, 1996 10:45 AM"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "alici",
"deger": "Portnoy, Sharon"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "konu",
"deger": "FW: NYC Rest. Contacts"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "tarih",
"deger": "Apr 4, 1996"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00143.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "Ce: Bayliss, Elissa; Culley, Liz; Young, Tracy Hi Sharon: Thad Elissa follow-up on obtaining info from the restaurants b"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "gonderen",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "alici",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "konu",
"deger": "Washington Post Story"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "tarih",
"deger": "April 29, 1998"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00124.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "Carothers, Paul Wednesday, April 29, 1998 6:07 PM Knuth, Kathy S. (MS Mail); Dennis, Darienne; Ostronic, John;"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "gonderen",
"deger": "ebh@caunxi.ctd.crnl.gov (B4 Harris)"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "alici",
"deger": "cvrGdsunix?.derd.crnl.gov"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "konu",
"deger": "Tobacco Funding Question"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "tarih",
"deger": "Tue, 26 Mar 1996 17:40:32 -0500"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "e_postalar",
"deger": "ebh@caunxi.ctd.crnl.gov, sikkavk@dsunix2.dsrd.ornl.cov, _vks@ornl.gov"
},
{
"belge": "images/test/email/test_email_00159.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "Ed_Harris,3/26/96 4:40 PM,Tobacco Funding Question ce: sikkavk@dsunix2.dsrd.ornl.cov vee,"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "gonderen",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "alici",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "konu",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "tarih",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00178.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "52021 9188 yO 8 Buunjoeynuey spuesa \"YO \"BuURDeINUEN SUSHPOUIERD MUAANG"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "gonderen",
"deger": "Date:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "alici",
"deger": "co:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "konu",
"deger": "To:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "tarih",
"deger": "Subject:"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "e_postalar",
"deger": "None"
},
{
"belge": "images/test/email/test_email_00100.png",
"tur": "email",
"alan": "govde_ozeti",
"deger": "co: 2071557599 Note for Knuff, Ketrina"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "tarih",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "firma",
"deger": "PHILIP MORRIS MANAGEMENT CORPORATION"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00263.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "1723.58"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "tarih",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "firma",
"deger": "WoRsN"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00241.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "tarih",
"deger": "1/24/89"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "firma",
"deger": "PROMOTIONAL MARKETING, INC"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00296.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "$25,500, $17,400,, $37,950"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "tarih",
"deger": "02/18/94"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "firma",
"deger": "Brand Precielon Services, Inc. 1so0a761 it"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "1,130.00"
},
{
"belge": "images/test/invoice/test_invoice_00218.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "10.00 , 50.00 , 500.00 , 10.00 , 22.00"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "fatura_no",
"deger": "None"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "tarih",
"deger": "November 8 00"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "firma",
"deger": "ANYNEX Company"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "toplam_tutar",
"deger": "$956.83"
},
{
"belge": "images/test/invoice/test_invoice_00298.png",
"tur": "invoice",
"alan": "tutarlar",
"deger": "$956.83, 9638.67 , 5638.67 , $5.10, 313.06"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "BIOGRAPHICAL SKETCH"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "universite",
"deger": "Tel-Aviv University, Tel-Aviv, Israel B.Sc. [1966-1969 Biology"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "Tel-Aviv University, Tel-Aviv, Israel Ph.D. [1973-1977 Endocrinology of Reproductic:"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "telefon",
"deger": "30502-30509"
},
{
"belge": "images/test/resume/test_resume_00341.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "EDWARD DIALA"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "universite",
"deger": "Harvard University Special Student in Biology 1972-4"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "Institution Subject & Degree Year"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "telefon",
"deger": "50145702"
},
{
"belge": "images/test/resume/test_resume_00323.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "ANN KATHERINE DALY"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "universite",
"deger": "BA in Biochemistry (1978, University of Dublin); PhD in Biochemistry (1982,"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "telefon",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00356.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "Cecilia M. Giachelli Research"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "universite",
"deger": "University of Washington, Seattle, WA Phb, 1987 Pharmacolo"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "American Association for the Advancement of Science"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "telefon",
"deger": "50304562"
},
{
"belge": "images/test/resume/test_resume_00374.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "ad_soyad",
"deger": "Pose Bs"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "universite",
"deger": "Metabolism, Baylor College of Medicine"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "bolum_derece",
"deger": "Key personnel typical wil include all Indviduls wih doctoral or other professional degrees, btn some projects wil includ"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "e_posta",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "telefon",
"deger": "None"
},
{
"belge": "images/test/resume/test_resume_00301.png",
"tur": "resume",
"alan": "beceriler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "Printed i Great Brin"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "DAVID 3"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00046.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "What the Physician Should Know About Tobacco"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "WALTER A. BASTEDO, tobaccos"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00057.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "Sondereudh us: eitge to Tabklordhng\" Band 8- Heb 7 Ohtake 976"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "Cytrel, Dav Git Vidroy"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00030.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "PN LEE, w+ FIR RED Ntel cit Gees"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "LEE, GP Se, Sretrcewn, Wie Lis"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00012.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "baslik",
"deger": "TOBACCO SHOKE AND THE PULMONARY ALVEOLAR MACROPHAGE"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "yazarlar",
"deger": "DRATH, P. DAVIES, M. L. KARNOVSKY, Beth Israel Hospital Boston, Teceresrtct"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "ozet",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "anahtar_kelimeler",
"deger": "None"
},
{
"belge": "images/test/scientific_publication/test_scientific_publication_00051.png",
"tur": "scientific_publication",
"alan": "kaynakca_var",
"deger": "False"
}
]

ev = pd.DataFrame(ROWS)
ev["deger"] = ev["deger"].replace({"None": "— (bulunamadı)"}).str.replace(r"\s+", " ", regex=True)
docs = list(dict.fromkeys(ev["belge"]))          # sıra korunur
done = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame(columns=["belge", "tur", "alan", "deger", "karar"])
print(f"{len(docs)} belge, {len(ev)} alan · daha önce değerlendirilen: {done['belge'].nunique()} belge")

## 2. Değerlendirme ekranı

Hücreyi çalıştır; ekran aşağıda açılır. Resim küçük gelirse **Resmi büyüt** düğmesine bas.

In [ ]:
import ipywidgets as w
import traceback
from IPython.display import display

CHOICES = ["Doğru", "Yanlış", "Kaçırıldı", "Belgede yok"]
state = {"i": next((k for k, d in enumerate(docs) if d not in set(done["belge"])), len(docs)), "big": False}
screen = w.VBox()          # ekran tek bir kutu; düğmeye basınca içeriği değiştirilir (Colab'da güvenilir yöntem)

def save_doc(belge, picks):
    global done
    sub = ev[ev.belge == belge].copy()
    sub["karar"] = [p.value for p in picks]
    done = pd.concat([done[done.belge != belge], sub], ignore_index=True)
    done.to_csv(OUT, index=False)

def render():
    i = state["i"]
    if i >= len(docs):
        screen.children = [w.HTML(f"<h3>Bitti! {done['belge'].nunique()} belge değerlendirildi.</h3>"
                                  "Sonuçlar için aşağıdaki 3. bölümdeki hücreyi çalıştır."),
                           back_btn()]
        return
    belge = docs[i]
    sub = ev[ev.belge == belge].reset_index(drop=True)
    prev = done[done.belge == belge].set_index("alan")["karar"].to_dict() if len(done) else {}
    img = Image.open(DATA / belge).convert("L")
    buf = io.BytesIO(); img.save(buf, format="PNG")
    width = 900 if state["big"] else 520
    pic = w.Image(value=buf.getvalue(), format="png", layout=w.Layout(width=f"{width}px"))
    head = w.HTML(f"<h3>Belge {i + 1} / {len(docs)} · {TR.get(sub.tur[0], sub.tur[0])} · {Path(belge).name}</h3>")
    picks, field_rows = [], []
    for _, r in sub.iterrows():
        t = w.ToggleButtons(options=CHOICES, value=prev.get(r.alan) if prev.get(r.alan) in CHOICES else None,
                            style={"button_width": "92px"})
        picks.append(t)
        field_rows.append(w.VBox([w.HTML(f"<b>{r.alan}</b>: <span style='font-family:monospace'>{r.deger}</span>"), t],
                                 layout=w.Layout(margin="0 0 10px 0")))
    msg = w.HTML("")
    nxt = w.Button(description="Kaydet ve sonraki belge", button_style="success", layout=w.Layout(width="220px"))
    zoom = w.Button(description="Resmi küçült" if state["big"] else "Resmi büyüt", layout=w.Layout(width="140px"))

    def on_next(_):
        try:
            if any(p.value is None for p in picks):
                eksik = sum(p.value is None for p in picks)
                msg.value = f"<span style='color:#b00'>{eksik} alan seçilmedi. Her alan için bir düğmeye bas.</span>"
                return
            msg.value = "Kaydediliyor..."
            save_doc(belge, picks)
            state["i"] += 1
            render()
        except Exception:
            msg.value = "<pre style='color:#b00'>" + traceback.format_exc()[-1500:] + "</pre>"
    def on_zoom(_):
        state["big"] = not state["big"]; render()
    nxt.on_click(on_next); zoom.on_click(on_zoom)

    right = w.VBox(field_rows + [w.HBox([back_btn(), nxt]), msg], layout=w.Layout(padding="0 0 0 16px", width="460px"))
    if state["big"]:
        screen.children = [head, zoom, pic, right]
    else:
        screen.children = [head, w.HBox([w.VBox([pic, zoom]), right])]

def back_btn():
    b = w.Button(description="Önceki belge", layout=w.Layout(width="140px"))
    def on_back(_):
        state["i"] = max(0, state["i"] - 1); render()
    b.on_click(on_back)
    return b

render()
display(screen)

## 3. Sonuçlar

Değerlendirmeyi bitirince (ya da istediğin zaman) bu hücreyi çalıştır.

In [ ]:
done = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame()
if done.empty:
    print("Henüz değerlendirme yok.")
else:
    def score(g):
        c = g["karar"].value_counts()
        d, y, k, b = (int(c.get(x, 0)) for x in ["Doğru", "Yanlış", "Kaçırıldı", "Belgede yok"])
        return pd.Series({"alan": len(g), "doğru": d, "yanlış": y, "kaçırıldı": k, "belgede yok": b,
                          "precision": round(d / (d + y), 3) if d + y else None,   # çıkardığının ne kadarı doğru
                          "recall": round(d / (d + k), 3) if d + k else None,      # belgede olanın ne kadarını buldu
                          "doğruluk": round((d + b) / len(g), 3)})                 # tüm kararlar içinde doğru davranma
    by_type = done.groupby("tur").apply(score)
    by_type.loc["TOPLAM"] = score(done)
    by_field = done.groupby(["tur", "alan"]).apply(score)
    print(f"Değerlendirilen: {done['belge'].nunique()} belge, {len(done)} alan\n")
    print("Türe göre:"); display(by_type)
    print("Alana göre:"); display(by_field)
    by_field.to_csv(ROOT / "cikarim_sonuclari_v2.csv")
    by_type.to_csv(ROOT / "cikarim_sonuclari_tur_v2.csv")
    print("Kaydedildi:", ROOT / "cikarim_sonuclari_v2.csv")

| Ölçüt | Anlamı |
|---|---|
| **Precision** | Sistemin bir şey çıkardığı alanların ne kadarı doğru |
| **Recall** | Belgede gerçekten olan bilgilerin ne kadarını sistem bulabildi |
| **Doğruluk** | "Belgede yok" doğru tespitleri de sayılınca, alanların ne kadarında sistem doğru davrandı |